In [7]:
import json
from pathlib import Path
from typing import Dict, List, Any
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 256

# ✅ local input + output
NODES_PATH = Path("personal.json")
OUT_PATH = Path("personal_embeddings.json")

def load_nodes() -> List[Dict[str, Any]]:
    data = json.loads(NODES_PATH.read_text(encoding="utf-8"))
    return list(data.values()) if isinstance(data, dict) else data

def extract_embedding_keys(nodes: List[Dict[str, Any]]) -> List[str]:
    keys: List[str] = []
    for n in nodes:
        k = n.get("embedding_key") or n.get("description") or n.get("text")
        if isinstance(k, str) and k.strip():
            keys.append(k.strip())

    # dedupe while preserving order
    seen = set()
    uniq = []
    for k in keys:
        if k not in seen:
            seen.add(k)
            uniq.append(k)
    return uniq

def main():
    if not NODES_PATH.exists():
        raise FileNotFoundError(f"Can't find {NODES_PATH.resolve()}")

    nodes = load_nodes()
    keys = extract_embedding_keys(nodes)
    print(f"Loaded nodes: {len(nodes)} | unique embedding keys: {len(keys)}")

    model = SentenceTransformer(MODEL_NAME)
    embeddings: Dict[str, List[float]] = {}

    for i in range(0, len(keys), BATCH_SIZE):
        batch = keys[i:i+BATCH_SIZE]
        vecs = model.encode(
            batch,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=False
        )
        for k, v in zip(batch, vecs):
            embeddings[k] = v.tolist()
        print(f"Embedded {min(i+BATCH_SIZE, len(keys))}/{len(keys)}")

    OUT_PATH.write_text(json.dumps(embeddings, indent=2), encoding="utf-8")
    print("Done. Wrote:", OUT_PATH.resolve())

if __name__ == "__main__":
    main()

Loaded nodes: 60 | unique embedding keys: 60


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 454.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]

Embedded 60/60
Done. Wrote: C:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\personal_embeddings.json


In [9]:
import json, math
from datetime import datetime
from pathlib import Path
from sentence_transformers import SentenceTransformer

NODES_PATH = Path("personal.json")
EMBS_PATH  = Path("personal_embeddings.json")

MODEL_NAME = "all-MiniLM-L6-v2"
TOP_K = 10

HALF_LIFE_HOURS = 24.0
W_RECENCY = 1.0
W_IMPORTANCE = 1.0
W_RELEVANCE = 1.0
REL_MIN = 0.62

TYPE_W = {"thought": 1.0, "chat": 1.0, "event": 1.0}

def parse_dt(s: str) -> datetime:
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(str(s), fmt)
        except:
            pass
    return datetime.fromisoformat(str(s))

def cosine(a, b) -> float:
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

def recency(created: datetime, now: datetime) -> float:
    dt_h = max(0.0, (now - created).total_seconds() / 3600.0)
    return 2 ** (-dt_h / HALF_LIFE_HOURS)

def importance(poignancy) -> float:
    p = float(poignancy or 0.0)
    return min(1.0, max(0.0, p / 10.0))

def load_nodes():
    data = json.loads(NODES_PATH.read_text(encoding="utf-8"))
    nodes = list(data.values()) if isinstance(data, dict) else data
    for n in nodes:
        n["_id"] = n.get("node_count") or n.get("node_id")
        n["_type"] = n.get("type")
        n["_t"] = parse_dt(n.get("created"))
        n["_txt"] = (n.get("description") or n.get("text") or "").strip()
        n["_k"] = (n.get("embedding_key") or n.get("description") or n.get("text") or "").strip()
    nodes.sort(key=lambda x: x["_t"])
    return nodes

def load_embs():
    e = json.loads(EMBS_PATH.read_text(encoding="utf-8"))
    return {k: v for k, v in e.items() if isinstance(v, list)}

def retrieve(nodes, embs, query: str, model, now=None, top_k=TOP_K):
    now = now or nodes[-1]["_t"]
    qvec = model.encode([query], normalize_embeddings=True)[0].tolist()

    out = []
    for n in nodes:
        vec = embs.get(n["_k"])
        if vec is None:
            continue

        rel = (cosine(qvec, vec) + 1) / 2
        if rel < REL_MIN:
            continue

        r = recency(n["_t"], now)
        imp = importance(n.get("poignancy"))
        tw = TYPE_W.get(n["_type"], 1.0)

        score = tw * (W_RECENCY * r + W_IMPORTANCE * imp + W_RELEVANCE * rel)
        out.append((score, r, imp, rel, n))

    out.sort(key=lambda x: x[0], reverse=True)
    return out[:top_k]

def build_llm_prompt(query: str, mem_rows):
    mem_lines = []
    for score, r, imp, rel, n in mem_rows:
        mem_lines.append(
            f"- id={n['_id']} type={n['_type']} time={n.get('created')} "
            f"(score={score:.3f}, imp={imp:.2f}, rel={rel:.2f}) :: {n['_txt']}"
        )

    return f"""You are a life-log assistant.

QUESTION:
{query}

RULES:
- Use ONLY the memories below.
- If evidence is insufficient, say exactly: Not enough evidence.
- Always cite evidence IDs in the Evidence section.

MEMORIES (retrieved):
{chr(10).join(mem_lines)}

OUTPUT FORMAT:
Answer: <2-5 sentences>
Evidence:
- id=<...>: <short quote or paraphrase>
- id=<...>: <short quote or paraphrase>
"""

if __name__ == "__main__":
    nodes = load_nodes()
    embs = load_embs()
    model = SentenceTransformer(MODEL_NAME)
    now = nodes[-1]["_t"]

    QUESTIONS = [
    "What personal record did Alex Chen hit on bench press, and where did he train?",
    "What options did Alex and Priya compare for the production RAG vector database, and what action did they agree to take?",
    "Across Alex’s reflections and chats, what consistent strategy is emerging to reduce chatbot hallucinations while managing cost and GPU limits?",
]

for i, QUESTION in enumerate(QUESTIONS, 1):
    mem_rows = retrieve(nodes, embs, QUESTION, model, now=now, top_k=TOP_K)
    prompt = build_llm_prompt(QUESTION, mem_rows)

    print("\n" + "=" * 90)
    print(f"PROMPT {i}/5 (copy-paste into ChatGPT/Gemini)")
    print("=" * 90)
    print(prompt)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 471.57it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



PROMPT 1/5 (copy-paste into ChatGPT/Gemini)
You are a life-log assistant.

QUESTION:
What personal record did Alex Chen hit on bench press, and where did he train?

RULES:
- Use ONLY the memories below.
- If evidence is insufficient, say exactly: Not enough evidence.
- Always cite evidence IDs in the Evidence section.

MEMORIES (retrieved):
- id=45 type=thought time=2025-02-28 19:00:00 (score=2.502, imp=0.70, rel=0.89) :: Alex Chen is reflecting on his gym progress this month and feeling proud about hitting a 225 lb bench press and a 315 lb squat, both personal records
- id=51 type=thought time=2025-02-28 12:30:00 (score=2.226, imp=0.80, rel=0.67) :: Alex Chen is reflecting on how chain-of-thought prompting significantly improved the accuracy of the classification model from 78% to 91% in yesterday's experiments
- id=44 type=event time=2025-02-28 20:00:00 (score=2.218, imp=0.60, rel=0.67) :: Alex Chen is watching Andrej Karpathy's YouTube lecture on building GPT from scratch and takin